## Part 3: ORM with SQLAlchemy

In [1]:
!pip install sqlalchemy matplotlib

In [2]:
# Import SQLAlchemy
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, ForeignKey, Table
from sqlalchemy.orm import sessionmaker, relationship, declarative_base
import datetime
import os

# Delete old database file so every run starts clean
if os.path.exists('ecommerce.db'):
    os.remove('ecommerce.db')
    print("Old database deleted.")
else:
    print("No existing database found — starting fresh.")


No existing database found — starting fresh.


In [3]:
# Database configuration
Base = declarative_base() # Classes 
engine = create_engine('sqlite:///ecommerce.db') # Database connection
Session = sessionmaker(bind=engine) # Session factory
session = Session() # Session instance

# Confirm the database connection
print("Database created successfully")


Database created successfully


In [4]:
# Many-to-Many association table between Orders and Products
order_product_table = Table(
    'order_product', Base.metadata,
    Column('order_id', Integer, ForeignKey('orders.order_id')),
    Column('product_id', Integer, ForeignKey('products.product_id')),
    Column('quantity', Integer)
)

print("Order-Product association table defined")

Order-Product association table defined


In [5]:
class Customer(Base):
    __tablename__ = 'customers'

    customer_id = Column(Integer, primary_key=True)
    name = Column(String)
    email = Column(String)
    created_at = Column(DateTime, default=datetime.datetime.utcnow)

    # One-to-Many: one customer can have many orders
    orders = relationship('Order', back_populates='customer')

    def __repr__(self):
        return f"<Customer(id={self.customer_id}, name='{self.name}', email='{self.email}')>"

In [ ]:
class Order(Base):
    __tablename__ = 'orders'

    order_id = Column(Integer, primary_key=True)
    customer_id = Column(Integer, ForeignKey('customers.customer_id'))
    order_date = Column(DateTime, default=datetime.datetime.utcnow)

    # Many-to-One - each order belongs to one customer
    customer = relationship('Customer', back_populates='orders')

    # Many-to-Many - orders to and from products through the join table
    products = relationship('Product', secondary=order_product_table, back_populates='orders')

    def __repr__(self):
        return f"<Order(id={self.order_id}, customer_id={self.customer_id}, date={self.order_date})>"

In [ ]:
class Product(Base):
    __tablename__ = 'products'

    product_id = Column(Integer, primary_key=True)
    name = Column(String)
    price = Column(Float)

    # Many-to-Many - products to and from orders through the join table
    orders = relationship('Order', secondary=order_product_table, back_populates='products')

    def __repr__(self):
        return f"<Product(id={self.product_id}, name='{self.name}', price=${self.price})>"

In [8]:
Base.metadata.create_all(engine)

from sqlalchemy import inspect
print("\nAll tables created successfully!")
print("Tables in database:", inspect(engine).get_table_names())


All tables created successfully!
Tables in database: ['customers', 'order_product', 'orders', 'products']


In [ ]:
# 3 Customers 
customer1 = Customer(name="Alice Johnson",  email="alice@example.com")
customer2 = Customer(name="Bob Smith",      email="bob@example.com")
customer3 = Customer(name="Charlie Brown",  email="charlie@example.com")

session.add_all([customer1, customer2, customer3])
session.commit() # Save customers to the database

print("Customers inserted:")
print(customer1)
print(customer2)
print(customer3)

Customers inserted:
<Customer(id=1, name='Alice Johnson', email='alice@example.com')>
<Customer(id=2, name='Bob Smith', email='bob@example.com')>
<Customer(id=3, name='Charlie Brown', email='charlie@example.com')>


In [ ]:
# 5 Products 
product1 = Product(name="Laptop",      price=1200.00)
product2 = Product(name="Smartphone",  price=800.00)
product3 = Product(name="Headphones",  price=150.00)
product4 = Product(name="Monitor",     price=300.00)
product5 = Product(name="Keyboard",    price=50.00)

session.add_all([product1, product2, product3, product4, product5])
session.commit()

print("Products inserted:")
for p in [product1, product2, product3, product4, product5]:
    print(p)

Products inserted:
<Product(id=1, name='Laptop', price=$1200.0)>
<Product(id=2, name='Smartphone', price=$800.0)>
<Product(id=3, name='Headphones', price=$150.0)>
<Product(id=4, name='Monitor', price=$300.0)>
<Product(id=5, name='Keyboard', price=$50.0)>


In [11]:
order1 = Order(customer_id=customer1.customer_id, products=[product1, product3])
# Alice buys a Laptop + Headphones

order2 = Order(customer_id=customer1.customer_id, products=[product2, product5])
# Alice also buys a Smartphone + Keyboard

order3 = Order(customer_id=customer2.customer_id, products=[product4])
# Bob buys a Monitor

order4 = Order(customer_id=customer3.customer_id, products=[product2, product3, product5])
# Charlie buys Smartphone + Headphones + Keyboard

order5 = Order(customer_id=customer2.customer_id, products=[product1, product4])
# Bob also buys a Laptop + Monitor (5th order to meet the minimum)

session.add_all([order1, order2, order3, order4, order5])
session.commit()

print("Orders inserted:")
for o in [order1, order2, order3, order4, order5]:
    print(o)

Orders inserted:
<Order(id=1, customer_id=1, date=2026-03-25 22:06:47.808692)>
<Order(id=2, customer_id=1, date=2026-03-25 22:06:47.808696)>
<Order(id=3, customer_id=2, date=2026-03-25 22:06:47.808697)>
<Order(id=4, customer_id=3, date=2026-03-25 22:06:47.808697)>
<Order(id=5, customer_id=2, date=2026-03-25 22:06:47.808698)>


/var/folders/4_/hvxs2p4n3_n407gz6szqh7jw0000gn/T/ipykernel_35694/1589248394.py:7: SAWarning: Object of type <Order> not in session, add operation along 'Product.orders' won't proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  order3 = Order(customer_id=customer2.customer_id, products=[product4])
/var/folders/4_/hvxs2p4n3_n407gz6szqh7jw0000gn/T/ipykernel_35694/1589248394.py:10: SAWarning: Object of type <Order> not in session, add operation along 'Product.orders' won't proceed (This warning originated from the Session 'autoflush' process, which was invoked automatically in response to a user-initiated operation. Consider using ``no_autoflush`` context manager if this warning happened while initializing objects.)
  order4 = Order(customer_id=customer3.customer_id, products=[product2, product3

In [12]:
print("=" * 50)
print("ALL CUSTOMERS IN DATABASE:")
for c in session.query(Customer).all():
    print(f"  {c.name} ({c.email}) — {len(c.orders)} order(s)")

print("\nALL PRODUCTS IN DATABASE:")
for p in session.query(Product).all():
    print(f"  {p.name} — ${p.price}")

print("\nALL ORDERS IN DATABASE:")
for o in session.query(Order).all():
    product_names = [p.name for p in o.products]
    print(f"  Order {o.order_id} by Customer {o.customer_id}: {product_names}")

print("=" * 50)
print("Data insertion complete and verified!")

ALL CUSTOMERS IN DATABASE:
  Alice Johnson (alice@example.com) — 2 order(s)
  Bob Smith (bob@example.com) — 2 order(s)
  Charlie Brown (charlie@example.com) — 1 order(s)

ALL PRODUCTS IN DATABASE:
  Laptop — $1200.0
  Smartphone — $800.0
  Headphones — $150.0
  Monitor — $300.0
  Keyboard — $50.0

ALL ORDERS IN DATABASE:
  Order 1 by Customer 1: ['Laptop', 'Headphones']
  Order 2 by Customer 1: ['Smartphone', 'Keyboard']
  Order 3 by Customer 2: ['Monitor']
  Order 4 by Customer 3: ['Smartphone', 'Headphones', 'Keyboard']
  Order 5 by Customer 2: ['Laptop', 'Monitor']
Data insertion complete and verified!


### Queries

In [13]:
# (a) Retrieve all orders for a given customer
def get_customer_orders(customer_name):
    customer = session.query(Customer).filter_by(name=customer_name).first()
    if customer:
        print(f"\nOrders for {customer.name}:")
        for order in customer.orders:
            print(f"  Order ID: {order.order_id}, Date: {order.order_date}")
    else:
        print("Customer not found.")

get_customer_orders("Alice Johnson")


Orders for Alice Johnson:
  Order ID: 1, Date: 2026-03-25 22:06:47.808692
  Order ID: 2, Date: 2026-03-25 22:06:47.808696


In [14]:
# (b) Retrieve all products in a specific order
def get_order_products(order_id):
    order = session.query(Order).filter_by(order_id=order_id).first()
    if order:
        print(f"\nProducts in Order {order_id}:")
        for product in order.products:
            print(f"  {product.name} - ${product.price}")
    else:
        print("Order not found.")

get_order_products(1)


Products in Order 1:
  Laptop - $1200.0
  Headphones - $150.0


In [15]:
# (c) Calculate total revenue per customer
def get_total_spent_per_customer():
    results = session.query(Customer, Order).join(Order).all()
    customer_spending = {}
    for customer, order in results:
        total_spent = sum([product.price for product in order.products])
        if customer.name in customer_spending:
            customer_spending[customer.name] += total_spent
        else:
            customer_spending[customer.name] = total_spent

    print("\nTotal Revenue per Customer:")
    for customer, total in customer_spending.items():
        print(f"  {customer}: ${total:.2f}")
    return customer_spending

spending_data = get_total_spent_per_customer()


Total Revenue per Customer:
  Alice Johnson: $2200.00
  Bob Smith: $1800.00
  Charlie Brown: $1000.00
